# Chapitre 5 : formater la sortie et parler à la place de Claude

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# Nouvel argument pour le texte de pré-remplissage, chaîne vide par défaut
def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Cours

**Claude peut formater sa réponse de très nombreuses façons.** Il suffit de le lui demander !

L'une d'elles consiste à utiliser des balises XML pour isoler la réponse de tout texte superflu. Vous savez déjà que les balises XML rendent votre prompt plus clair et plus facile à analyser pour Claude. Vous pouvez aussi demander à Claude **d'utiliser des balises XML pour rendre sa réponse plus claire et plus facile à exploiter** par un humain.

### Exemples

Vous vous souvenez du « problème du préambule du poème », résolu au chapitre 2 en demandant à Claude de supprimer complètement le préambule ? On peut obtenir un résultat similaire en **demandant à Claude de placer le poème entre balises XML**.

In [ ]:
# Contenu de la variable
ANIMAL = "Rabbit"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Quel intérêt ? Avec une réponse **entre balises XML, l'utilisateur final peut récupérer de façon fiable le poème, et uniquement le poème, grâce à un petit programme qui extrait le contenu situé entre les balises**.

Une extension de cette technique consiste à **placer la première balise XML dans le tour `assistant`**. Mettre du texte dans le tour `assistant` revient à dire à Claude qu'il a déjà commencé à répondre et qu'il doit continuer à partir de là. Cette technique s'appelle « parler à la place de Claude » ou « pré-remplir la réponse de Claude » (prefill).

Ci-dessous, nous l'appliquons avec la balise ouvrante `<haiku>`. Remarquez que Claude reprend directement là où nous nous sommes arrêtés.

In [ ]:
# Contenu de la variable
ANIMAL = "Cat"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Pré-remplissage de la réponse de Claude
PREFILL = "<haiku>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Claude maîtrise aussi d'autres formats de sortie, notamment le `JSON`. Pour imposer une sortie JSON (pas de façon déterministe, mais presque), vous pouvez pré-remplir la réponse de Claude avec l'accolade ouvrante `{`.

In [ ]:
# Contenu de la variable
ANIMAL = "Cat"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Pré-remplissage de la réponse de Claude
PREFILL = "{"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Voici un exemple avec **plusieurs variables d'entrée dans le même prompt ET une spécification du format de sortie, le tout à l'aide de balises XML**.

In [ ]:
# Première variable d'entrée
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Deuxième variable d'entrée
ADJECTIVE = "olde english"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Pré-remplissage de la réponse de Claude (désormais une f-string avec variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

#### Leçon bonus

Si vous appelez Claude via l'API, vous pouvez passer la balise XML fermante dans le paramètre `stop_sequences` pour que Claude s'arrête dès qu'il l'émet. Cela économise de l'argent et du temps de génération, en supprimant les remarques de conclusion que Claude ajouterait après la réponse qui vous intéresse.

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 5.1 - Steph Curry, le meilleur de tous les temps](#exercice-51---steph-curry-le-meilleur-de-tous-les-temps)
- [Exercice 5.2 - Deux haïkus](#exercice-52---deux-haïkus)
- [Exercice 5.3 - Deux haïkus, deux animaux](#exercice-53---deux-haïkus-deux-animaux)

### Exercice 5.1 - Steph Curry, le meilleur de tous les temps
Obligé de choisir, Claude désigne Michael Jordan comme meilleur joueur de basket de tous les temps. Peut-on l'amener à choisir quelqu'un d'autre ?

Modifiez la variable `PREFILL` pour **obliger Claude à argumenter en détail que le meilleur joueur de tous les temps est Stephen Curry**. Essayez de ne modifier que `PREFILL`, c'est l'objet de l'exercice.

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Pré-remplissage de la réponse de Claude
PREFILL = ""

# Obtenir la réponse de Claude
response = get_completion(PROMPT, prefill=PREFILL)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_5_1_hint; print(exercise_5_1_hint)

### Exercice 5.2 - Deux haïkus
Modifiez le `PROMPT` ci-dessous à l'aide de balises XML pour que Claude écrive deux haïkus sur l'animal au lieu d'un seul. On doit voir clairement où finit un poème et où commence l'autre.

In [ ]:
# Contenu de la variable
ANIMAL = "cats"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Pré-remplissage de la réponse de Claude
PREFILL = "<haiku>"

# Obtenir la réponse de Claude
response = get_completion(PROMPT, prefill=PREFILL)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_5_2_hint; print(exercise_5_2_hint)

### Exercice 5.3 - Deux haïkus, deux animaux
Modifiez le `PROMPT` ci-dessous pour que **Claude écrive deux haïkus sur deux animaux différents**. Utilisez `{ANIMAL1}` pour la première variable et `{ANIMAL2}` pour la seconde.

In [ ]:
# Première variable d'entrée
ANIMAL1 = "Cat"

# Deuxième variable d'entrée
ANIMAL2 = "Dog"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Obtenir la réponse de Claude
response = get_completion(PROMPT)

# Fonction de correction de l'exercice
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_5_3_hint; print(exercise_5_3_hint)

### Bravo !

Si vous avez résolu tous les exercices jusqu'ici, vous pouvez passer au chapitre suivant. Bon prompting !

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Contenu de la variable
ANIMAL = "Rabbit"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Contenu de la variable
ANIMAL = "Cat"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Pré-remplissage de la réponse de Claude
PREFILL = "<haiku>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Contenu de la variable
ANIMAL = "Cat"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Pré-remplissage de la réponse de Claude
PREFILL = "{"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Première variable d'entrée
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Deuxième variable d'entrée
ADJECTIVE = "olde english"

# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Pré-remplissage de la réponse de Claude (désormais une f-string avec variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))